In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)

con = duckdb.connect("../data/credit.duckdb", read_only=True)
df = con.execute("SELECT * FROM clients").df()
con.close()

df.shape

(30000, 25)

In [2]:
df.head(10)


,client_id,limit_bal,sex,education,marriage,age,pay_1,pay_2,pay_3,pay_4,pay_5,pay_6,bill_amt1,bill_amt2,bill_amt3,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,default
0,1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0
5,6,50000,1,1,2,37,0,0,0,0,0,0,64400,57069,57608,19394,19619,20024,2500,1815,657,1000,1000,800,0
6,7,500000,1,1,2,29,0,0,0,0,0,0,367965,412023,445007,542653,483003,473944,55000,40000,38000,20239,13750,13770,0
7,8,100000,2,2,2,23,0,-1,-1,0,0,-1,11876,380,601,221,-159,567,380,601,0,581,1687,1542,0
8,9,140000,2,3,1,28,0,0,2,0,0,0,11285,14096,12108,12211,11793,3719,3329,0,432,1000,1000,1000,0
9,10,20000,1,3,2,35,-2,-2,-2,-2,-1,-1,0,0,0,0,13007,13912,0,0,0,13007,1122,0,0


In [3]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
client_id,30000.0,15000.500000,8660.398374,1.0,7500.75,15000.5,22500.25,30000.0
limit_bal,30000.0,167484.322667,129747.661567,10000.0,50000.00,140000.0,240000.00,1000000.0
sex,30000.0,1.603733,0.489129,1.0,1.00,2.0,2.00,2.0
education,30000.0,1.853133,0.790349,0.0,1.00,2.0,2.00,6.0
marriage,30000.0,1.551867,0.521970,0.0,1.00,2.0,2.00,3.0
age,30000.0,35.485500,9.217904,21.0,28.00,34.0,41.00,79.0
pay_1,30000.0,-0.016700,1.123802,-2.0,-1.00,0.0,0.00,8.0
pay_2,30000.0,-0.133767,1.197186,-2.0,-1.00,0.0,0.00,8.0
pay_3,30000.0,-0.166200,1.196868,-2.0,-1.00,0.0,0.00,8.0
pay_4,30000.0,-0.220667,1.169139,-2.0,-1.00,0.0,0.00,8.0


In [4]:
for col in ["education", "marriage", "pay_1"]:
    print(col, df[col].value_counts().sort_index().to_dict())

education {0: 14, 1: 10585, 2: 14030, 3: 4917, 4: 123, 5: 280, 6: 51}
marriage {0: 54, 1: 13659, 2: 15964, 3: 323}
pay_1 {-2: 2759, -1: 5686, 0: 14737, 1: 3688, 2: 2667, 3: 322, 4: 76, 5: 26, 6: 11, 7: 9, 8: 19}


In [5]:
df.groupby("pay_1")["default"].agg(["count", "mean"]).round(3)

,count,mean
pay_1,,
-2,2759,0.132
-1,5686,0.168
0,14737,0.128
1,3688,0.339
2,2667,0.691
3,322,0.758
4,76,0.684
5,26,0.500
6,11,0.545


In [6]:
from contextlib import redirect_stdout

with open("../eda_output.txt", "w") as f, redirect_stdout(f):
    print("=== Cell 1: shape ===")
    print(df.shape)

    print("\n=== Cell 2: first 10 rows ===")
    print(df.head(10).to_string())

    print("\n=== Cell 3: describe ===")
    print(df.describe().T.round(1).to_string())

    print("\n=== Cell 4: categorical codes ===")
    for col in ["education", "marriage", "pay_1"]:
        print(col, df[col].value_counts().sort_index().to_dict())

    print("\n=== Cell 5: default rate by pay_1 ===")
    print(df.groupby("pay_1")["default"].agg(["count", "mean"]).round(3).to_string())

    print("\n=== Cell 6: negative bills / over limit ===")
    print("negative bill_amt1:", (df["bill_amt1"] < 0).sum())
    print("bill_amt1 over limit:", (df["bill_amt1"] > df["limit_bal"]).sum())

print("saved to eda_output.txt")

saved to eda_output.txt


## EDA findings and cleaning decisions

### Findings
1. **Size and target:** 30,000 clients, 25 columns, no nulls. Default rate 22.1%, so the classes are imbalanced (about 1 in 5).
2. **Undocumented education codes:** 0 (14), 5 (280) and 6 (51) are not in the data dictionary (1 to 4).
3. **Undocumented marriage code:** 0 (54).
4. **Repayment codes −2, −1 and 0** all behave as "not late". Default rates are 13.2%, 16.8% and 12.8%.
5. **Lateness drives risk:** the default rate rises to 33.9% at 1 month late and 69.1% at 2 months, against 12.8% for "0".
6. **Sparse high delays:** pay_1 values of 4 to 8 have only 9 to 76 clients each, so their rates are noisy.
7. **Negative bills:** 590 clients had a negative September bill (credit balance). The minimum is −339,603 (bill_amt6).
8. **Over-limit:** 2,115 clients (about 7%) had a September bill above their credit limit.
9. **Skewed payments:** the median pay_amt1 is 2,100, but the mean is 5,664 and the max is 873,552.
10. **Delinquency worsens over time:** the mean repayment status rises from −0.29 (April) to −0.02 (September).
11. **Plausible ranges:** age 21 to 79, limit 10,000 to 1,000,000 (right-skewed: mean 167k, median 140k).

### Cleaning plan
1. Map education 0, 5 and 6 to 4 ("other").
2. Map marriage 0 to 3 ("other"). Used only for the fairness check, not in the model.
3. Delay = max(pay, 0), so −2, −1 and 0 become "not late". Cap at 3 for logistic regression.
4. Keep negative-bill rows. In ratios, a bill of 0 or less means nothing owed (ratio = 1).
5. Handle skew with ratios to the limit (utilisation) and log1p for raw amounts.
6. Drop no rows.
7. Exclude sex and marriage from the model on purpose (fairness).
8. For logistic regression only: clip util_recent, util_avg and util_max at their 99th percentile, computed on the training set only (no rows dropped; gradient boosting uses raw values).